In [2]:
import numpy as np
import pandas as pd

# --- Load ---
data = pd.read_excel(
    "C:/Users/ADMIN/Documents/VEES_Bulletin_Hackathon/Datasets/combined_reports.xlsx"
)

# --- Disease/Condition: fill "Other" with free text ---
data.loc[data["Disease/Condition"].eq("Other"), "Disease/Condition"] = data["Other Disease"]

# --- Disease Control Method: safely replace "Other" token with free text ---
def fix_control_method(row):
    method = row["Disease Control Method"]
    other = row["Other Control Method"]
    if pd.isna(method):
        return method
    if pd.isna(other):
        return method
    # split on comma in case it's a multi-select field, swap exact "Other" tokens
    parts = [p.strip() for p in str(method).split(",")]
    parts = [str(other) if p == "Other" else p for p in parts]
    return ", ".join(parts)

data["Disease Control Method"] = data.apply(fix_control_method, axis=1)

# --- Drop the now-redundant "Other" columns (safe even if already dropped) ---
data.drop(columns=["Other Disease", "Other Control Method"], inplace=True, errors="ignore")

# --- Drop Number Affected* and Location* columns into a new frame ---
data2 = data.drop(
    columns=[c for c in data.columns if c.startswith("Number Affected") or c.startswith("Location")]
)

print(data2.columns.tolist())

# --- Harmonise "Other Species" free text ---
species_map = {
    "rabbit": "Rabbits", "rabbits": "Rabbits", "rabit": "Rabbits", "rabits": "Rabbits", "rabbit kitten": "Rabbits",
    "chicken": "Chicken", "broiler": "Chicken", "broilers": "Chicken", "poultry": "Chicken", "avian": "Chicken",
    "duck": "Ducks", "ducks": "Ducks",
    "turkey": "Turkeys",
    "dog": "Dogs", "dogs": "Dogs", "canine": "Dogs", "puppies": "Dogs",
    "goat": "Goats", "goats": "Goats", "shoat": "Goats", "shoats": "Goats",
    "kid": "Goats", "kids": "Goats", "goat lambs": "Goats", "sheep and goat": "Goats",
    "lamb": "Sheep", "lambs": "Sheep", "sheep and goats": "Sheep",
    "calf": "Cattle","Goats,Sheep And Cattle": "Cattle",
    "camel": "Camel", "horse": "Horse", "fish": "Fish", "insecta": "Insect",
    "donkey": "Donkey", "zebra": "Zebra", "eland": "Eland",
}

data2["Other Species"] = (
    data2["Other Species"].astype("string").str.strip().str.lower()
)
data2["Other Species"] = data2["Other Species"].replace(species_map)
data2["Other Species"] = data2["Other Species"].str.strip().str.title()

# --- Check harmonised results ---
data2["Other Species"].value_counts()

['Source Form Version', 'Submitter UUID', 'Submitter Username', 'Submitter Name', 'Submitter Organization', 'Submitter Role', 'Id', 'Date of Report', 'Date of Start of Outbreak/Event', 'Date Event Reported', 'County', 'Sub-County', 'Ward', 'Locality', 'Species Affected', 'Other Species', 'Production System', 'Abortion', 'Sudden Death', 'Death', 'Hemorrhagic Signs', 'Neurologic signs', 'Animal Bites', 'Respiratory Signs', 'Oral/Foot Lesions', 'Cutaneous/Skin Lesions', 'Gastrointestinal tract syndromes', 'Other Syndromes', 'Number at Risk', 'Number Sick / Bitten', 'Number Dead', 'Disease/Condition', 'Nature of Diagnosis', 'Test Used', 'Number of Humans Affected (If zoonosis)', 'Number Slaughtered', 'Number Destroyed', 'Number Vaccinated', 'Disease Control Method', 'Longitude', 'Latitude', 'Number Sick']


Other Species
Rabbits                   59
Chicken                   15
Goats                     10
Dogs                       8
Ducks                      7
Turkeys                    4
Sheep                      4
Horse                      3
Fish                       2
Camel                      2
Human                      2
Geese                      2
Pigs                       2
Insect                     1
Donkey                     1
Flamingoes                 1
Ostrich                    1
Horse/Equine               1
Cattle                     1
Horses                     1
Elephant                   1
Leopard                    1
Man                        1
Feline( Cat)               1
Goats,Sheep And Cattle     1
Zebra                      1
Eland                      1
Name: count, dtype: Int64

In [3]:
data2.columns

Index(['Source Form Version', 'Submitter UUID', 'Submitter Username',
       'Submitter Name', 'Submitter Organization', 'Submitter Role', 'Id',
       'Date of Report', 'Date of Start of Outbreak/Event',
       'Date Event Reported', 'County', 'Sub-County', 'Ward', 'Locality',
       'Species Affected', 'Other Species', 'Production System', 'Abortion',
       'Sudden Death', 'Death', 'Hemorrhagic Signs', 'Neurologic signs',
       'Animal Bites', 'Respiratory Signs', 'Oral/Foot Lesions',
       'Cutaneous/Skin Lesions', 'Gastrointestinal tract syndromes',
       'Other Syndromes', 'Number at Risk', 'Number Sick / Bitten',
       'Number Dead', 'Disease/Condition', 'Nature of Diagnosis', 'Test Used',
       'Number of Humans Affected (If zoonosis)', 'Number Slaughtered',
       'Number Destroyed', 'Number Vaccinated', 'Disease Control Method',
       'Longitude', 'Latitude', 'Number Sick'],
      dtype='str')

In [4]:
data2.isnull().sum().sort_values(ascending=False)

Other Species                              65024
Date Event Reported                        64727
Test Used                                  64619
Number Sick / Bitten                       53893
Death                                      53893
Number Sick                                11265
Number of Humans Affected (If zoonosis)    10959
Latitude                                    6821
Longitude                                   6807
Production System                           4999
Submitter Organization                      4005
Disease/Condition                             99
Nature of Diagnosis                           86
Submitter Role                                19
Submitter Name                                 0
Submitter UUID                                 0
Source Form Version                            0
Submitter Username                             0
Date of Start of Outbreak/Event                0
Date of Report                                 0
Id                  

In [5]:
data2["Species Affected"].value_counts()

Species Affected
Cattle     33817
Goats      16104
Sheep       5796
Dogs        2933
Camel       2422
Chicken     1997
Donkey       922
Pigs         815
Other        134
Cats         124
Bees          94
Name: count, dtype: int64

In [6]:
# Merge: where "Species Affected" is "Other", pull in the harmonised "Other Species" value
data2.loc[data2["Species Affected"].eq("Other"), "Species Affected"] = data2["Other Species"]

# Drop the now-redundant column
data2.drop(columns=["Other Species"], inplace=True, errors="ignore")

# Check results
data2["Species Affected"].value_counts()

Species Affected
Cattle                    33818
Goats                     16114
Sheep                      5800
Dogs                       2941
Camel                      2424
Chicken                    2012
Donkey                      923
Pigs                        817
Cats                        124
Bees                         94
Rabbits                      59
Ducks                         7
Turkeys                       4
Horse                         3
Fish                          2
Human                         2
Geese                         2
Insect                        1
Flamingoes                    1
Ostrich                       1
Horse/Equine                  1
Horses                        1
Elephant                      1
Leopard                       1
Man                           1
Feline( Cat)                  1
Goats,Sheep And Cattle        1
Zebra                         1
Eland                         1
Name: count, dtype: int64

In [7]:
data2["Species Affected"] = data2["Species Affected"].replace(
    {"Goats,Sheep And Cattle": "Goats"}
)

data2["Species Affected"].value_counts()

Species Affected
Cattle          33818
Goats           16115
Sheep            5800
Dogs             2941
Camel            2424
Chicken          2012
Donkey            923
Pigs              817
Cats              124
Bees               94
Rabbits            59
Ducks               7
Turkeys             4
Horse               3
Fish                2
Human               2
Geese               2
Insect              1
Flamingoes          1
Ostrich             1
Horse/Equine        1
Horses              1
Elephant            1
Leopard             1
Man                 1
Feline( Cat)        1
Zebra               1
Eland               1
Name: count, dtype: int64

In [8]:
data2.isnull().sum().sort_values(ascending=False)

Date Event Reported                        64727
Test Used                                  64619
Number Sick / Bitten                       53893
Death                                      53893
Number Sick                                11265
Number of Humans Affected (If zoonosis)    10959
Latitude                                    6821
Longitude                                   6807
Production System                           4999
Submitter Organization                      4005
Disease/Condition                             99
Nature of Diagnosis                           86
Submitter Role                                19
Submitter UUID                                 0
Submitter Username                             0
Source Form Version                            0
Submitter Name                                 0
Date of Start of Outbreak/Event                0
Date of Report                                 0
Id                                             0
Sub-County          

In [11]:
data2["Date Event Reported"] = pd.to_datetime(data2["Date Event Reported"], errors="coerce")
data2["Number of Humans Affected (If zoonosis)"] = data2["Number of Humans Affected (If zoonosis)"].astype("Int64")
data2["Latitude"] = data2["Latitude"].astype("float64")
data2["Longitude"] = data2["Longitude"].astype("float64")

for col in ["Test Used", "Production System", "Submitter Organization",
            "Disease/Condition", "Nature of Diagnosis", "Submitter Role"]:
    data2[col] = data2[col].astype("category")

In [13]:
data2.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 65158 entries, 0 to 65157
Data columns (total 41 columns):
 #   Column                                   Non-Null Count  Dtype         
---  ------                                   --------------  -----         
 0   Source Form Version                      65158 non-null  str           
 1   Submitter UUID                           65158 non-null  str           
 2   Submitter Username                       65158 non-null  str           
 3   Submitter Name                           65158 non-null  str           
 4   Submitter Organization                   61153 non-null  category      
 5   Submitter Role                           65139 non-null  category      
 6   Id                                       65158 non-null  str           
 7   Date of Report                           65158 non-null  str           
 8   Date of Start of Outbreak/Event          65158 non-null  str           
 9   Date Event Reported                      431 non-n

In [14]:
data2.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 65158 entries, 0 to 65157
Data columns (total 41 columns):
 #   Column                                   Non-Null Count  Dtype         
---  ------                                   --------------  -----         
 0   Source Form Version                      65158 non-null  str           
 1   Submitter UUID                           65158 non-null  str           
 2   Submitter Username                       65158 non-null  str           
 3   Submitter Name                           65158 non-null  str           
 4   Submitter Organization                   61153 non-null  category      
 5   Submitter Role                           65139 non-null  category      
 6   Id                                       65158 non-null  str           
 7   Date of Report                           65158 non-null  str           
 8   Date of Start of Outbreak/Event          65158 non-null  str           
 9   Date Event Reported                      431 non-n

In [ ]:
numeric_null_cols = [
    "Number Sick / Bitten",
    "Number Sick",
    "Number of Humans Affected (If zoonosis)"
]

for col in numeric_null_cols:
    data2[col] = data2[col].fillna(0).astype("Int64")

geo_cols = ["Latitude", "Longitude"]

for col in geo_cols:
    data2[col] = data2[col].where(data2[col].notna(), pd.NA).astype("float64")

In [18]:
data2.isnull().sum().sort_values(ascending=False)

Date Event Reported                        64727
Test Used                                  64619
Death                                      53893
Latitude                                    6821
Longitude                                   6807
Production System                           4999
Submitter Organization                      4005
Disease/Condition                             99
Nature of Diagnosis                           86
Submitter Role                                19
Date of Start of Outbreak/Event                0
Id                                             0
Date of Report                                 0
Submitter UUID                                 0
Submitter Username                             0
Source Form Version                            0
Submitter Name                                 0
Abortion                                       0
Sudden Death                                   0
Hemorrhagic Signs                              0
Sub-County          